<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/04%20-%20Logica%20Proposicional%20Conectivos%20e%20Permissivos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 04 - Notebook: Implementação de Conectivos Lógicos e Permissivos de Partida

Neste notebook implementamos as funções de avaliação lógica proposicional completas (AND, OR, NOT, XOR, IMPLICATION, BICONDITIONAL) e construímos os blocos de permissivos de movimentação (*Start Permissives*) e intertravamento contínuo (Trip) para os atuadores do AGV Logístico.

In [1]:
from typing import Dict
import pandas as pd
import itertools

# Operadores Fundamentais da Lógica Proposicional
def NOT(p: bool) -> bool:
    return not p

def AND(p: bool, q: bool) -> bool:
    return p and q

def OR(p: bool, q: bool) -> bool:
    return p or q

def XOR(p: bool, q: bool) -> bool:
    return p ^ q

def IMPLIES(p: bool, q: bool) -> bool:
    return (not p) or q

def IFF(p: bool, q: bool) -> bool:
    return p == q

print("Operadores lógicos proposicionais carregados com sucesso.")

Operadores lógicos proposicionais carregados com sucesso.


## Bloco Lógico de Permissivo do Motor de Tração M-301 (AGV)

O permissivo garante que o motor só avance se o modo estiver correto, o operador for detectado (IA) e não houver nenhuma falha ativa.

In [2]:
def permissivo_motor_M301(c1_operador: bool, d1_colisao: bool, g1_gas: bool, b1_bateria: bool, s1_emergencia: bool, autonomo_mode: bool, manual_mode: bool) -> Dict[str, bool]:
    # Condição de modo exclusivo (Autônomo XOR Manual)
    modo_valido = XOR(autonomo_mode, manual_mode)

    # Condição combinada de permissivo (AND em série)
    permissivo = (c1_operador and
                  NOT(d1_colisao) and
                  NOT(g1_gas) and
                  NOT(b1_bateria) and
                  NOT(s1_emergencia) and
                  modo_valido)

    # Condição de trip imediato (Bloqueio Contínuo) - De Morgan
    trip = NOT(c1_operador) or d1_colisao or g1_gas or b1_bateria or s1_emergencia

    return {
        'Permissivo_Habilitado': permissivo,
        'Trip_Ativo': trip,
        'Modo_Valido': modo_valido
    }

# Teste com diferentes cenários operacionais e de campo
cenarios = [
    {"cenario": "Operação Segura (Autônomo)", "args": (True, False, False, False, False, True, False)},
    {"cenario": "Perda de Visão do Operador", "args": (False, False, False, False, False, True, False)},
    {"cenario": "Operador na Zona de Colisão", "args": (True, True, False, False, False, True, False)},
    {"cenario": "Vazamento de Gás NH3 Detectado", "args": (True, False, True, False, False, True, False)},
    {"cenario": "Bateria em Nível Crítico", "args": (True, False, False, True, False, True, False)},
    {"cenario": "Parada de Emergência Ativa", "args": (True, False, False, False, True, True, False)},
    {"cenario": "Conflito de Modo (Auto e Manual juntos)", "args": (True, False, False, False, False, True, True)},
]

resultados = []
for c in cenarios:
    res = permissivo_motor_M301(*c["args"])
    resultados.append({
        "Cenário": c["cenario"],
        "Permissivo": res["Permissivo_Habilitado"],
        "Trip Ativo": res["Trip_Ativo"],
        "Modo Válido": res["Modo_Valido"]
    })

pd.DataFrame(resultados)

,Cenário,Permissivo,Trip Ativo,Modo Válido
0,Operação Segura (Autônomo),True,False,True
1,Perda de Visão do Operador,False,True,True
2,Operador na Zona de Colisão,False,True,True
3,Vazamento de Gás NH3 Detectado,False,True,True
4,Bateria em Nível Crítico,False,True,True
5,Parada de Emergência Ativa,False,True,True
6,Conflito de Modo (Auto e Manual juntos),False,False,False


## Geração Automática de Tabela-Verdade para Validação Exaustiva

Geração do espaço de estados completo para comprovar que em todas as combinações lógicas possíveis de campo, a tração se comporta de forma estritamente segura.

In [3]:
variaveis = ['c1_operador', 'd1_colisao', 'g1_gas', 'b1_bateria', 's1_emergencia']
tabela = []

# Iterando sobre as 32 combinações lógicas possíveis (2^5)
for combo in itertools.product([False, True], repeat=len(variaveis)):
    st = dict(zip(variaveis, combo))
    res = permissivo_motor_M301(st['c1_operador'], st['d1_colisao'], st['g1_gas'], st['b1_bateria'], st['s1_emergencia'], True, False)
    row = {**st, 'Permissivo': res['Permissivo_Habilitado'], 'Trip': res['Trip_Ativo']}
    tabela.append(row)

df_tv = pd.DataFrame(tabela)
print(f"Total de combinações avaliadas: {len(df_tv)}")
print(f"Combinações seguras que liberam a tração (Permissivo=True): {df_tv['Permissivo'].sum()}")
print("\nPrimeiras 8 combinações testadas:")
print(df_tv.head(8))

Total de combinações avaliadas: 32
Combinações seguras que liberam a tração (Permissivo=True): 1

Primeiras 8 combinações testadas:
   c1_operador  d1_colisao  g1_gas  b1_bateria  s1_emergencia  Permissivo  \
0        False       False   False       False          False       False   
1        False       False   False       False           True       False   
2        False       False   False        True          False       False   
3        False       False   False        True           True       False   
4        False       False    True       False          False       False   
5        False       False    True       False           True       False   
6        False       False    True        True          False       False   
7        False       False    True        True           True       False   

   Trip  
0  True  
1  True  
2  True  
3  True  
4  True  
5  True  
6  True  
7  True  
